<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 MiniMax H3-Image: 16 MP Text to Image + Editing</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition | Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>2K images in about 15 seconds, up to 16 MP, plus multi-image editing | MiniMax H3-Image 20B INT8 + pDMD 2-step | Powered by Wan2GP</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

**[MiniMax H3-Image](https://huggingface.co/MiniMaxAI/MiniMax-H3-Image)** is MiniMax's open image model: it generates images of up to **16 megapixels** (4096x4096 and every aspect ratio up to 21:9) in just 6 steps, follows long detailed prompts closely, and edits or combines your own pictures.

- ⚡ **Turbo**: a 2K image in 2 steps, about 15 seconds on 2 x T4
- 🖼️ **Text to image**: 2K (4 MP) or 16 MP PNGs
- ✏️ **Editing**: add up to 4 pictures and describe the finished result

**Quick start:** run **Step 1** and **Step 2**, open the link Step 2 prints, click the first example under **📚 Examples** at the bottom of the page and press **🎨 Generate**.

## ✍️ How to write prompts

The model follows **long, complete descriptions** much better than short requests. Describe the finished image as if you were captioning it:

1. **What the image is:** a documentary photograph, an oil painting, a 3D render, an anime frame, a poster.
2. **The scene:** the main subject, what it is doing, the setting, a few supporting details.
3. **Composition and light:** the viewpoint (overhead, close-up, low angle), where things sit in the frame, the light source and its direction, the colors.
4. **Text:** put every word to render in quotes and say where it goes.

**Editing:** open **🖼️ Pictures to edit**, add 1 to 4 pictures and refer to them as `<Picture 1>`, `<Picture 2>`... Describe what changes **and list what stays exactly the same** (faces, poses, background, light). *Keep Picture 1's proportions* gives the output the shape of the first picture; *Use the selected size* keeps the size you picked.

## 🚀 Speed settings

- **⚡ Turbo: 2 steps** (default): the pDMD few-step LoRA. Sharp and coherent; graphic prompts can invent pseudo-text.
- **⚖️ Balanced: 4 steps**: more fine detail; better for quoted text and flat graphics.
- **💎 Quality: 6 steps**: the model's own schedule, as MiniMax released it. The same seed gives a different picture than Turbo.

## ⏱️ How long it takes on 2 x T4

| Image | ⚡ Turbo | ⚖️ Balanced | 💎 Quality |
|---|---|---|---|
| 2K (2048x2048) | **~15 s** | ~25 s | ~35 s |
| 2K edit, 1 picture | ~30 s | ~50 s | ~70 s |
| 16 MP (4736x3552) | ~1 min | ~2 min | ~3 min |

The first edit of a session takes about 15 s longer (one-time kernel setup for the longer input).

**⚡ Attention** (next to the seed): *Fast* uses Comfy Kitchen's INT8 attention for the T4; *Exact* is the full-precision reference. **Text encoder** (first line of Step 1): *fast* is a distilled Qwen3-VL-4B made for still images; *full* is the original Qwen3-VL-32B (about 10 s more per new prompt).

> **License:** MiniMax H3-Image is released under the [MiniMax H3-Image Community License](https://huggingface.co/MiniMaxAI/MiniMax-H3-Image/blob/main/LICENSE). Label results as AI-generated and only edit pictures you have the rights to.

## 📦 Step 1: Environment, Wan2GP and model downloads

Installs Wan2GP and the packages it needs, then downloads ~29 GB of weights (about 4-6 minutes; ~38 GB with the full text encoder). Files you attach as a Kaggle Dataset are used instead of downloading.

In [ ]:
# Step 1: environment, pinned Wan2GP, dependencies and model downloads

# 👉 Text encoder: "fast" (distilled Qwen3-VL-4B, prompts in ~2 s, made for still images) or "full" (Qwen3-VL-32B, ~12 s)
TEXT_ENCODER = "fast"
import os
import sys
import time
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~20 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_h3_image.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# ---- Settings (every source is locked to one version) ----
WAN2GP_COMMIT = "38758d3c73366422efcf609abb63cab8385f3552"   # deepbeepmeep/Wan2GP, 12 Nov 2026 (the version this notebook is built on)
H3_REVISION = "b2292a57700a7cb15a27265d4493416534b20c1f"      # DeepBeepMeep/MiniMax-H3 (INT8 transformer, text encoder, VAE)
EXAMPLES_REVISION = "8ced60068623def5b74b4d10558117cb61d6482d"  # MiniMaxAI/MiniMax-H3-Image (official examples)
MODELS_DIR = "/kaggle/tmp/h3_image_models" if os.path.isdir("/kaggle") else os.path.abspath("h3_image_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of the weights in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if TEXT_ENCODER not in ("fast", "full"):
    raise SystemExit(f'❌ Unknown TEXT_ENCODER "{TEXT_ENCODER}". Set it to "fast" or "full".')
try:
    gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
except Exception:
    gpus = []
ram_gb = psutil.virtual_memory().total / 1024**3
if len(gpus) < 2:
    print("❌ This notebook needs 2 GPUs: Settings → Accelerator → GPU T4 x2")
print(f"✅ Text encoder: {TEXT_ENCODER} | {len(gpus)} x {gpus[0] if gpus else 'no GPU'} | RAM {ram_gb:.0f} GB")

# ---- 1. Wan2GP, fetched by commit (never a branch) ----
t0 = time.time()
if not os.path.isdir("Wan2GP/.git"):
    subprocess.run(["git", "init", "-q", "Wan2GP"], check=True)
    subprocess.run(["git", "-C", "Wan2GP", "remote", "add", "origin", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
subprocess.run(["git", "-C", "Wan2GP", "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT], check=True)
subprocess.run(["git", "-C", "Wan2GP", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits on re-runs
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]} ({time.time() - t0:.0f}s)")

# ---- 2. Dependencies: only what the H3-Image path imports ----
# (Wan2GP's full requirements.txt pins an onnxruntime nightly that is not on PyPI.)
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
                "comfy-kitchen==0.2.35", "transformers==4.54.0", "diffusers==0.36.0", "gguf==0.17.1", "optimum-quanto==0.2.7",
                "accelerate>=1.1.1", "gradio==5.29.0", "huggingface_hub[hf_xet]", "hf_xet>=1.5.2", "sentencepiece", "soundfile",
                "ffmpeg-python", "imageio", "imageio-ffmpeg"], check=True)
print(f"✅ Dependencies installed ({time.time() - t0:.0f}s)")

# ---- 3. Model weights (parallel; files from an attached Kaggle Dataset are reused) ----
H3, EXAMPLES = "DeepBeepMeep/MiniMax-H3", "MiniMaxAI/MiniMax-H3-Image"
DIT = (H3, "MiniMax-H3-Image-pruned_rank8_int8_convrot.safetensors", 21.07, H3_REVISION)
BIG = [DIT,
       (H3, "minimax_h3_image/minimax_h3_image_vae_fp16.safetensors", 1.17, H3_REVISION),
       (H3, "loras/minimax_h3_pdmd_2nfe_bf16.safetensors", 1.38, H3_REVISION)]  # pDMD few-step LoRA (2 / 4 steps)
if TEXT_ENCODER == "fast":
    TE = "P-H3-Image-Qwen3-VL-4B"
    BIG.append((H3, f"{TE}/P-H3-Image-Qwen3-VL-4B_int8_convrot.safetensors", 5.12, H3_REVISION))
    SMALL = [(H3, f"{TE}/{name}", 0.0, H3_REVISION) for name in ("config.json", "tokenizer.json", "tokenizer_config.json")]
else:
    TE = "Qwen3-VL-32B-Instruct"
    BIG.append((H3, f"{TE}/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf", 14.58, H3_REVISION))
    SMALL = [(H3, f"{TE}/{name}", 0.0, H3_REVISION) for name in
             ("config.json", "tokenizer.json", "tokenizer_config.json", "vocab.json", "preprocessor_config.json", "chat_template.json")]
SMALL += [(EXAMPLES, f"examples/{name}", 0.0, EXAMPLES_REVISION) for name in
          ("revised_prompt.t2i.txt", "revised_prompt.edit.txt", "reference1.png")]
os.makedirs(MODELS_DIR, exist_ok=True)

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            attached.setdefault(f, os.path.join(root, f))


def fetch(item):
    from huggingface_hub import hf_hub_download
    repo, filename, size_gb, revision = item
    dest = os.path.join(MODELS_DIR, filename)
    if os.path.exists(dest):
        return f"✓ {filename}"
    source = attached.get(os.path.basename(filename)) if size_gb else None
    if source and os.path.getsize(source) > 0:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir=MODELS_DIR)
            return f"✅ {filename}" + (f" ({size_gb:.1f} GB)" if size_gb else "")
        except Exception as e:
            if attempt == 3:
                raise
            say(f"⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


todo = [item for item in BIG if not os.path.exists(os.path.join(MODELS_DIR, item[1])) and os.path.basename(item[1]) not in attached]
need_gb = sum(item[2] for item in todo)
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
if need_gb > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need_gb:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA (bytes on disk, which hf_xet fills as it goes)
    base = folder_bytes(MODELS_DIR)
    total = max(need_gb * 1e9, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        for job in [pool.submit(fetch, item) for item in BIG + SMALL]:
            job.result()
finally:
    done.set()
    time.sleep(1.1)
if need_gb:
    say(f"✅ {need_gb:.1f} GB downloaded in {fmt_time(time.time() - t0)} (avg {need_gb * 1e3 / max(time.time() - t0, 1):.0f} MB/s)")
shutil.rmtree(os.path.join(MODELS_DIR, ".cache"), ignore_errors=True)

# Settings for Step 2's engine (also saved to a file, so Step 2 can be re-run alone after a kernel restart)
import json
H3I_SETTINGS = {"H3I_MODELS_DIR": MODELS_DIR, "H3I_DIT_FILE": DIT[1], "H3I_EXAMPLES_DIR": os.path.join(MODELS_DIR, "examples"),
                "H3I_TEXT_ENCODER": TEXT_ENCODER}
os.environ.update(H3I_SETTINGS)
with open("h3_image_settings.json", "w") as f:
    json.dump(H3I_SETTINGS, f)
print(f"✅ All weights ready. Run Step 2 (disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB)")

## 🚀 Step 2: Engine + Gradio UI

Starts the engine: model load and a warm-up image (~3-4 min), then the **Cloudflare** and **Gradio** public links. Open either one. If the Cloudflare link is still warming up, it appears in the cell a minute later.

**In the app:** write a detailed prompt → pick a size and speed → 🎨 **Generate**. For edits, add pictures under 🖼️ *Pictures to edit*. The progress bar at the top of the page keeps working if the connection drops, and *📂 Load latest image* fetches the result at any time.

In [ ]:
# Step 2: Engine + branded Gradio UI
# Writes run_h3_image.py, then runs it: model load (~3 min) → public links.
import codecs
import os
import re
import sys
import subprocess

RUN_SCRIPT_CODE = r'''"""MiniMax H3-Image text-to-image and image editing engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_h3_image.py and runs it. It drives Wan2GP's MiniMax H3-Image code:

* DiT: MiniMax H3-Image Pruned 20B in INT8 ConvRot, no guidance. 2 or 4 steps with the pDMD few-step LoRA (run live in
  FP16 on top of the INT8 weights), or the model's own 6-step schedule (one output head per step).
* Prompts: Pruna's distilled Qwen3-VL-4B text encoder (or the full Qwen3-VL-32B GGUF), on the exact path.
* Every INT8 linear runs on Comfy Kitchen's INT8 tensor-core GEMM (W8A8 with the ConvRot Hadamard rotation).
* Tensor-parallel DiT: every block is split in half across the two T4s (attention heads and FFN columns), so both GPUs
  compute every block at the same time and only exchange the block's partial outputs. Nothing is streamed per step.
  Each GPU's half is queued from a CPU thread of its own, so neither GPU waits for the other's kernel launches.
* Attention: Comfy Kitchen's INT8 attention for the T4 by default, or exact PyTorch SDPA (FP16) in the UI.
* The text encoder and the image VAE are pinned in RAM and run by MMGP on GPU 1, unloaded before denoising.
"""
import copy
import faulthandler
import functools
import gc
import glob
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
import warnings

warnings.filterwarnings("ignore")
faulthandler.enable(file=sys.__stderr__, all_threads=True)  # a hard crash (segfault, abort) prints where it happened
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

WAN2GP_DIR = os.path.abspath("Wan2GP")
WORK_DIR = os.path.dirname(WAN2GP_DIR)
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
MODELS_DIR = os.environ.get("H3I_MODELS_DIR", "/kaggle/tmp/h3_image_models")
EXAMPLES_DIR = os.environ.get("H3I_EXAMPLES_DIR", os.path.join(MODELS_DIR, "examples"))
DIT_FILE = os.environ.get("H3I_DIT_FILE", "MiniMax-H3-Image-pruned_rank8_int8_convrot.safetensors")
TEXT_ENCODER = os.environ.get("H3I_TEXT_ENCODER", "fast")  # "fast": distilled Qwen3-VL-4B, "full": Qwen3-VL-32B
TE_PATHS = {"fast": ("P-H3-Image-Qwen3-VL-4B", "P-H3-Image-Qwen3-VL-4B_int8_convrot.safetensors"),
            "full": ("Qwen3-VL-32B-Instruct", "qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf")}
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

import numpy as np
import psutil
import torch
from PIL import Image

import gradio as gr


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _ram_in_use():
    """total - available: unlike psutil's 'used', this counts pinned (page-locked) memory, which Linux files as shared."""
    vm = psutil.virtual_memory()
    return f"{_gb(vm.total - vm.available)} / {_gb(vm.total)}"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites it in place).
    A finished phase just clears its line (minimal logs: callers print one summary line where it matters). Step-based phases (denoising) interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between callbacks."""

    WIDTH, LINE = 24, 104
    WATCHED = {"decode": 1800, "stitch": 600, "audio": 600, "save": 600}  # phase -> seconds before a stack dump

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.drawn = 0  # characters of the live line on screen
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            if self.phase is not None and self.phase["key"] == key:
                return
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            if key in self.WATCHED:  # a phase that should take seconds: dump every thread's stack if it hangs
                faulthandler.dump_traceback_later(self.WATCHED[key], file=sys.__stderr__)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.flush()

    def abort(self):
        """End the live line of a phase that failed (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.write(f"  ❌ {p['label']} failed after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<16} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.1f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        text = self._render()
        self.drawn = max(self.drawn, len(text), self.LINE)
        self.out.write("\r" + text.ljust(self.drawn))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.drawn + "\r")
            self.visible = False
            self.drawn = 0

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)



if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise SystemExit("❌ This notebook needs Kaggle's 'GPU T4 x2' accelerator (Settings → Accelerator → GPU T4 x2).")

DIT_DEV = torch.device("cuda", 0)   # one half of every DiT block
MAIN_DEV = torch.device("cuda", 1)  # the other half + text encoder + VAEs + sampler state
torch.cuda.set_device(MAIN_DEV)     # Wan2GP / MMGP use the current device for "cuda"
DEVICES = (DIT_DEV, MAIN_DEV)

torch.backends.cuda.enable_flash_sdp(False)       # Turing (SM 7.5) has no flash kernel
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = False

_props = torch.cuda.get_device_properties(0)


def _on(dev):
    """torch.cuda.device(dev) for a CUDA device; nothing for the CPU."""
    import contextlib
    return torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()


# Turing has no BF16 convolution kernels. Convolutions that reach BF16 run in FP32.
if _props.major < 8:
    def _conv_fp32_on_turing(original):
        def forward(self, input, weight, bias):
            if input.is_cuda and input.dtype == torch.bfloat16:
                bias32 = None if bias is None else bias.float()
                return original(self, input.float(), weight.float(), bias32).to(torch.bfloat16)
            return original(self, input, weight, bias)
        return forward
    for _conv in (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d):
        _conv._conv_forward = _conv_fp32_on_turing(_conv._conv_forward)

# ==== Wan2GP runtime: quantization handlers + INT8 kernels (what wgp.py does at import) ====
# Wan2GP's shared utilities import rembg (background removal, which pulls in onnxruntime) at module level. This
# engine never removes backgrounds, so a stand-in module is registered when rembg is not installed.
import types as _types

try:
    import rembg  # noqa: F401
except Exception:
    _rembg = _types.ModuleType("rembg")

    def _no_rembg(*_args, **_kwargs):
        raise RuntimeError("Background removal (rembg) is not installed in this notebook")

    _rembg.remove = _rembg.new_session = _no_rembg
    sys.modules["rembg"] = _rembg

from mmgp import offload, quant_router

quant_router.unregister_handler(".fp8_quanto_bridge")
for _handler in ("shared.qtypes.int8_convrot", "shared.qtypes.gguf"):  # DiT, Qwen3-VL GGUF
    quant_router.register_handler(_handler)
from shared.qtypes import gguf as _gguf_handler

quant_router.register_file_extension("gguf", _gguf_handler)

from shared.kernels import int8_backend

try:
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using the PyTorch path: {e}")
# Kitchen splits each INT8 GEMM into row tiles that fit a 16 MB scratch buffer, which on a T4 means 64-row GEMMs
# for the 28672-wide FFN. 128 MB gives ~700-row tiles: far better tensor-core use for ~100 MB of extra scratch.
int8_backend._SCRATCH_BYTES = 128 * 1024 * 1024
INT8_LABEL = {"kitchen": "Comfy Kitchen INT8 tensor cores", "wgpkernels": "WGPKernels INT8", "triton": "Triton INT8 kernels"}.get(
    int8_backend._backend, "PyTorch INT8 fallback (slow)")
offload.shared_state["_attention"] = "sdpa"

from shared.utils import files_locator as fl

fl.set_checkpoints_paths([MODELS_DIR, "ckpts", "."])

import models.minimax_h3.minimax_h3_main as h3_main
from models.minimax_h3 import pipeline as h3_pipeline
from models.minimax_h3.minimax_h3_handler import family_handler
from models.minimax_h3.h3_image import (H3_IMAGE_PRUNED_ARCHITECTURE, H3_IMAGE_REFERENCE_SHORT_EDGE, H3_IMAGE_STEPS,
                                        H3_IMAGE_VAE_FILE, bound_short_edge, canvas_size)
from models.minimax_h3.pdd import MiniMaxH3ParallelHead

# ComfyUI-layout pruned checkpoints may also carry the AdaLN basis / mean they were compressed with. Wan2GP's model has
# no such tensors, so drop them before loading.
_orig_strip_wrappers = h3_main._strip_wrappers


def _strip_wrappers(state_dict, *args, **kwargs):
    for key in [k for k in state_dict if k.endswith(("adaln_basis", "adaln_mean"))]:
        state_dict.pop(key)
    return _orig_strip_wrappers(state_dict, *args, **kwargs)


h3_main._strip_wrappers = _strip_wrappers


# ==== Text encoder compute dtype ====
# Without llama.cpp kernels (no wheel for Kaggle's Python), each GGUF linear is dequantized to a dense weight and run
# as a normal matmul. Turing has no BF16 tensor cores, so a BF16 GEMM can be slower than plain FP32. Measure both on
# the real layer shape and keep the faster one (FP32 is also the more precise of the two).
def _gemm_seconds(dtype, m=512, k=5120, n=25600, reps=3):
    a = torch.randn(m, k, device=MAIN_DEV, dtype=dtype)
    b = torch.randn(n, k, device=MAIN_DEV, dtype=dtype)
    torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    t = time.time()
    for _ in range(reps):
        torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    return (time.time() - t) / reps


try:
    if TEXT_ENCODER == "fast":  # INT8 ConvRot weights: Wan2GP's BF16 compute dtype
        TE_DTYPE = torch.bfloat16
    else:
        _bf16_s, _fp32_s = _gemm_seconds(torch.bfloat16), _gemm_seconds(torch.float32)
        TE_DTYPE = torch.float32 if _fp32_s < _bf16_s else torch.bfloat16
except Exception:
    TE_DTYPE = torch.bfloat16
torch.cuda.empty_cache()
_orig_load_text_encoder = h3_main._load_text_encoder
h3_main._load_text_encoder = lambda filename, dtype, distilled=False: _orig_load_text_encoder(filename, TE_DTYPE, distilled)

# ==== Load the model ====
MODEL_TYPE = H3_IMAGE_PRUNED_ARCHITECTURE
_t_load = time.time()
BAR.start("load", "Loading models (~3 min)")
model_def = {"architecture": MODEL_TYPE}
model_def.update(family_handler.query_model_def(MODEL_TYPE, model_def))
model_def["qkv_splitting"] = False  # keep the fused QKV GEMM: bigger tiles, and the tensor-parallel split below
if TEXT_ENCODER == "fast":  # Pruna's Qwen3-VL-4B, distilled to give the 32B encoder's signal: prompts in about a second
    model_def["text_encoder_folder"] = TE_PATHS["fast"][0]
pipeline, _ = family_handler.load_model(
    model_filename=[os.path.join(MODELS_DIR, DIT_FILE)],
    model_type=MODEL_TYPE,
    base_model_type=MODEL_TYPE,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=os.path.join(MODELS_DIR, *TE_PATHS[TEXT_ENCODER]),
)
if not pipeline.h3_image:
    raise SystemExit("❌ The pipeline was not built for MiniMax H3-Image")
transformer = pipeline.transformer
transformer.cache = None
DISTILLED_TE = hasattr(pipeline.text_encoder, "text_connector")
if DISTILLED_TE != (TEXT_ENCODER == "fast"):
    raise SystemExit("❌ The text encoder did not load as selected in Step 1")

# The text encoder stays on the exact path: an LLM's activation outliers do not survive INT8 activation quantization,
# so its INT8 ConvRot weights are dequantized per layer (weight-only INT8) instead of going through Kitchen's W8A8 GEMM.
_orig_te_encode = pipeline.text_encoder.encode


def _exact_te_encode(*args, **kwargs):
    saved = int8_backend._backend, getattr(int8_backend, "_rotated_triton", None)
    int8_backend._backend = "pytorch"
    if hasattr(int8_backend, "_rotated_triton"):
        int8_backend._rotated_triton = None
    try:
        return _orig_te_encode(*args, **kwargs)
    finally:
        int8_backend._backend = saved[0]
        if hasattr(int8_backend, "_rotated_triton"):
            int8_backend._rotated_triton = saved[1]


pipeline.text_encoder.encode = _exact_te_encode

# Modules H3 pins to FP32 (patch projections, the per-step output heads, AdaLN curve projections) are converted here,
# since this transformer is not handed to MMGP's profile (which would otherwise apply the locks).
for _module in transformer.modules():
    _lock = getattr(_module, "_lock_dtype", None)
    if _lock is not None and isinstance(_module, (torch.nn.Linear, MiniMaxH3ParallelHead)) and any(
            p.dtype.is_floating_point for p in _module.parameters(recurse=False)):
        _module.to(_lock)

# ==== Tensor-parallel DiT: each block is split in half across the two T4s ====
# Within a block, GPU 0 computes attention heads 0-27 and FFN columns 0-7167, GPU 1 the other half. Both start from
# the same residual stream (kept as a replica on each GPU), and after attention and after the FFN each GPU sends its
# partial output projection to the other, so both GPUs hold the full sum. The weight slices follow the ConvRot
# layout exactly: QKV / FFN-up are split by output rows (Q, K, V are grouped), output / FFN-down by input columns on
# 256-wide rotation-group boundaries, so every INT8 GEMM computes precisely its share of the full product.
import models.minimax_h3.transformer as _h3_transformer
from models.minimax_h3.transformer import MLP as H3MLP, Attention as H3Attention, _gated_residual, _norm_modulate, _take
from shared.qtypes.int8_convrot import _rotate_activation

_tp_cache = {"rope": None, "rope_copy": None, "temb": None, "temb_copy": None, "src": None, "replica": None}


# >>> section:tp
class Int8Shard(torch.nn.Module):
    """One GPU's slice of an INT8 ConvRot linear (a set of output rows, or a range of input columns)."""

    def __init__(self, linear, device, rows=None, cols=None):
        super().__init__()
        data, scale = linear.weight._data, linear.weight._scale.reshape(-1)
        if rows is not None:
            data, scale = data.index_select(0, rows), scale.index_select(0, rows)
        if cols is not None:
            data = data[:, cols]
        # copy=True: own storage even on the same device (never a view into the memory-mapped checkpoint)
        self.register_buffer("qdata", data.contiguous().to(device, copy=True), persistent=False)
        self.register_buffer("qscale", scale.float().contiguous().to(device, copy=True), persistent=False)
        self.group = int(getattr(linear, "_convrot_group_size", 0) or 0)
        self.out_features, self.in_features = self.qdata.shape

    def forward(self, x):
        x2 = x.reshape(-1, self.in_features)
        if int8_backend.kitchen_enabled() and self.group == 256 and x2.is_cuda and self.out_features % 4 == 0:
            out = int8_backend._linear_impl(x2, self.qdata, self.qscale, None, True)
        else:  # reference path (no Kitchen): rotate, dequantize, matmul
            weight = self.qdata.to(x2.dtype) * self.qscale.to(x2.dtype)[:, None]
            out = torch.nn.functional.linear(_rotate_activation(x2, self.group), weight)
        return out.reshape(*x.shape[:-1], self.out_features)


def _copy_to(value, dev, slot):
    """Per-forward copies of temb / rope for GPU 0 (they are the same tensor object for all 50 blocks)."""
    if value is None:
        return None
    first = value[0] if isinstance(value, tuple) else value
    if first.device == dev:
        return value
    if _tp_cache[slot] is not first:
        _tp_cache[slot] = first
        _tp_cache[slot + "_copy"] = tuple(v.to(dev) for v in value) if isinstance(value, tuple) else value.to(dev)
    return _tp_cache[slot + "_copy"]


def _all_reduce_residual(hiddens, partials, gates, segments):
    """hidden += gate * (partial_0 + partial_1) on both GPUs."""
    incoming = [partials[1].to(DEVICES[0], non_blocking=True, copy=True),
                partials[0].to(DEVICES[1], non_blocking=True, copy=True)]
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            branch = partials[s].add_(incoming[s])
            out.append(_gated_residual([hiddens[s]], [gates[s]], [branch], segments))
    return out


# Long-sequence mode: the same block math on row slices. A normal block builds several full-length intermediates per GPU
# (modulated input, fused QKV, BF16 and FP16 copies of Q/K/V, the output-projection input, the exchanged partial sums):
# roughly 75-110 KB per token. For very long sequences that does not fit next to half of the model, so in this mode only
# what attention needs at full length is kept (the residual stream and FP16 Q/K/V, ~40 KB per token). Norms,
# projections, RoPE, the FFN and the GPU exchange run on 8192-row slices. Same arithmetic, a little more Python per block.
LONG_VIDEO = {"on": False, "rows": 8192}


def _row_chunks(n):
    step = LONG_VIDEO["rows"]
    return [(a, min(n, a + step)) for a in range(0, n, step)]


def _slice_segments(segments, a, b):
    """(start, stop, modality row) segments of the packed sequence, clipped to rows [a, b) and shifted to 0."""
    return [(max(s0, a) - a, min(s1, b) - a, row) for s0, s1, row in segments if min(s1, b) > max(s0, a)]


def _apply_rope(query, key, rope):
    """H3's split-half rotary embedding on [1, n, heads, dim] tensors, in place."""
    pairs = rope.shape[-2]
    cosine, sine = rope[..., 0], rope[..., 1]
    scratch = torch.empty_like(query[..., :pairs])
    for tensor in (query, key):
        first, second = tensor[..., :pairs], tensor[..., pairs:2 * pairs]
        scratch.copy_(first)
        first.mul_(cosine).addcmul_(second, sine, value=-1)
        second.mul_(cosine).addcmul_(scratch, sine)


def _sliced_attention(shard, hidden, mod, segments, rope):
    attn = shard.attn
    rows, heads, dim = hidden.shape[0], attn.heads, attn.head_dim
    inner = heads * dim
    if isinstance(rope, tuple):
        rope = rope[0]
    q, k, v = (torch.empty(1, rows, heads, dim, device=hidden.device, dtype=torch.float16) for _ in range(3))
    for a, b in _row_chunks(rows):
        h = _norm_modulate(shard.norm1, hidden[a:b], [mod[0], mod[1]], _slice_segments(segments, a, b))
        qkv = attn.qkv_proj(h)
        del h
        query, key, value = (t.reshape(1, b - a, heads, dim) for t in qkv.split(inner, dim=-1))
        query, key = attn.q_norm(query), attn.k_norm(key)
        if rope is not None:
            _apply_rope(query, key, rope[:, a:b])
        q[:, a:b].copy_(query)
        k[:, a:b].copy_(key)
        v[:, a:b].copy_(value)
        del qkv, query, key, value
    qkv_list = [q, k, v]
    del q, k, v
    out = _h3_transformer.pay_attention(qkv_list, recycle_q=True)[0]  # [rows, heads, dim] FP16
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(rows):
        partial[a:b] = attn.out_proj(out[a:b].reshape(b - a, inner).to(hidden.dtype))
    return partial


def _sliced_mlp(shard, hidden, mod, segments):
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(hidden.shape[0]):
        h = _norm_modulate(shard.norm2, hidden[a:b], [mod[3], mod[4]], _slice_segments(segments, a, b))
        partial[a:b] = shard.mlp([h])
    return partial


def _all_reduce_sliced(hiddens, partials, gates, segments):
    """_all_reduce_residual without full-length copies: the partial sums cross between GPUs slice by slice."""
    for a, b in _row_chunks(hiddens[0].shape[0]):
        incoming = [partials[1][a:b].to(DEVICES[0], non_blocking=True, copy=True),
                    partials[0][a:b].to(DEVICES[1], non_blocking=True, copy=True)]
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials[s][a:b].add_(incoming[s])
        del incoming
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            out.append(_gated_residual([hiddens[s]], [gates[s]], [partials[s]], segments))
    return out


# One CPU thread per GPU. With a single Python thread queuing both GPUs' kernels one after the other, anything that holds
# that thread up on one GPU (a full launch queue, a copy between the GPUs staged through host memory, an allocator flush)
# also holds back the other GPU's next kernels, and the other GPU sits idle. Each GPU has a thread of its own that queues
# its half of every block, and the two threads meet only where the math needs both halves: the two partial-sum exchanges
# per block. Same kernels in the same order on each GPU, so the result is
# bit-identical. The first transformer pass of the session runs on one thread (it compiles every kernel on both GPUs);
# the per-GPU threads take over from the second pass.
TP_THREADS = {"on": True, "warm": False}
TP_STATE = {"block": -1}  # index of the block being queued (read by the INT8 attention check)
_TP_BARRIER = threading.Barrier(len(DEVICES), timeout=1800)
_TP_POOLS = []


def _tp_pools():
    if not _TP_POOLS:
        from concurrent.futures import ThreadPoolExecutor
        for dev in DEVICES:
            init = (lambda d=dev: torch.cuda.set_device(d)) if dev.type == "cuda" else None
            _TP_POOLS.append(ThreadPoolExecutor(1, thread_name_prefix=f"tp{len(_TP_POOLS)}", initializer=init))
    return _TP_POOLS


def _tp_job(fn, s, grad, inference, args):
    try:
        with torch.inference_mode(inference), torch.set_grad_enabled(grad), _on(DEVICES[s]):
            return fn(s, *args)
    except BaseException:
        _TP_BARRIER.abort()  # the other GPU's thread stops waiting at its next exchange
        raise


def _run_on_both(fn, args_per_gpu):
    """fn(s, *args) on every GPU's own thread at once. Returns when both threads have queued their kernels, not when the
    GPUs are done with them."""
    if _TP_BARRIER.broken:
        _TP_BARRIER.reset()
    grad, inference = torch.is_grad_enabled(), torch.is_inference_mode_enabled()
    futures = [pool.submit(_tp_job, fn, s, grad, inference, args) for s, (pool, args) in enumerate(zip(_tp_pools(), args_per_gpu))]
    results, errors = [], []
    for future in futures:
        try:
            results.append(future.result())
        except BaseException as e:
            errors.append(e)
    if errors:  # the real error, not the other thread's broken barrier
        raise next((e for e in errors if not isinstance(e, threading.BrokenBarrierError)), errors[0])
    return results


def _exchange(s, hidden, partial, gate, segments, shared, sliced):
    """hidden += gate * (partial_0 + partial_1) on GPU s, from GPU s's thread. Both copies are queued (each on its source
    GPU's stream) before either thread adds into its own partial sum, so no copy can read a sum that was already changed."""
    other = 1 - s
    shared[s] = partial
    _TP_BARRIER.wait()  # both partial sums are queued
    for a, b in (_row_chunks(hidden.shape[0]) if sliced else [(0, hidden.shape[0])]):
        incoming = shared[other][a:b].to(DEVICES[s], non_blocking=True, copy=True)
        _TP_BARRIER.wait()  # both copies of these rows are queued
        partial[a:b].add_(incoming)
        del incoming
    shared[s] = None  # the other thread has queued its last read of it (before the barrier above)
    return _gated_residual([hidden], [gate], [partial], segments)


def _half_block(s, block, hidden, temb, rope, segments, shared, sliced):
    """GPU s's half of a DiT block, queued from GPU s's thread: the same kernels as TPBlock's one-thread path."""
    shard = block.shards[s]
    mod = shard.adaln_proj(temb)
    if sliced:
        partial = _sliced_attention(shard, hidden, mod, segments, rope)
    else:  # the modulated input is handed over in a list: attention frees it once Q, K and V exist
        partial = shard.attn([_norm_modulate(shard.norm1, hidden, [mod[0], mod[1]], segments)], rope=rope)
    hidden = _exchange(s, hidden, partial, mod[2], segments, shared, sliced)
    del partial
    if sliced:
        partial = _sliced_mlp(shard, hidden, mod, segments)
    else:
        partial = shard.mlp([_norm_modulate(shard.norm2, hidden, [mod[3], mod[4]], segments)])
    return _exchange(s, hidden, partial, mod[5], segments, shared, sliced)


class TPBlock(torch.nn.Module):
    def __init__(self, block, index=0):
        super().__init__()
        self.index = index
        attn, mlp = block.attn, block.mlp
        heads, head_dim = attn.heads, attn.head_dim
        inner, hidden = heads * head_dim, mlp.fc2.out_features
        ffn = mlp.fc2.in_features
        half_inner, half_ffn = inner // 2, ffn // 2
        if getattr(attn, "qkv_interleaved", False):
            raise RuntimeError("The tensor-parallel split expects grouped Q|K|V rows (quantized H3 checkpoints)")
        self.splits = []
        shards = []
        for s, dev in enumerate(DEVICES):
            h = torch.arange(s * half_inner, (s + 1) * half_inner)
            f = torch.arange(s * half_ffn, (s + 1) * half_ffn)
            qkv_rows = torch.cat((h, h + inner, h + 2 * inner))
            fc1_rows = torch.cat((f, f + ffn))  # SwiGLU: gate rows, then value rows
            out_cols = slice(s * half_inner, (s + 1) * half_inner)
            fc2_cols = slice(s * half_ffn, (s + 1) * half_ffn)
            self.splits.append({"qkv": qkv_rows, "fc1": fc1_rows, "out": out_cols, "fc2": fc2_cols})
            with torch.device("meta"):
                sattn = H3Attention(hidden, heads // 2, head_dim, attn.q_norm.eps, dtype=torch.bfloat16)
                smlp = H3MLP(hidden, half_ffn, dtype=torch.bfloat16)
            sattn.qkv_proj = Int8Shard(attn.qkv_proj, dev, rows=qkv_rows)
            sattn.out_proj = Int8Shard(attn.out_proj, dev, cols=out_cols)
            sattn.q_norm = copy.deepcopy(attn.q_norm).to(dev)
            sattn.k_norm = copy.deepcopy(attn.k_norm).to(dev)
            smlp.fc1 = Int8Shard(mlp.fc1, dev, rows=fc1_rows)
            smlp.fc2 = Int8Shard(mlp.fc2, dev, cols=fc2_cols)
            shard = torch.nn.Module()
            shard.attn, shard.mlp = sattn, smlp
            shard.norm1 = copy.deepcopy(block.norm1).to(dev)
            shard.norm2 = copy.deepcopy(block.norm2).to(dev)
            shard.adaln_proj = copy.deepcopy(block.adaln_proj).to(dev)
            shards.append(shard)
        self.shards = torch.nn.ModuleList(shards)

    @property
    def attn(self):  # H3's forward checks block.attn for q_proj / vdn
        return self.shards[1].attn

    def forward(self, x_list, temb, segments, rope, residual_signature_elements=0):
        if residual_signature_elements:
            raise RuntimeError("First Block Cache is not supported by the tensor-parallel DiT")
        hidden = _take(x_list)
        hidden = hidden if hidden.device == MAIN_DEV else hidden.to(MAIN_DEV)
        replica = _tp_cache["replica"] if _tp_cache["src"] is hidden else hidden.to(DIT_DEV, copy=True)
        hiddens = [replica, hidden]
        tembs = [_copy_to(temb, DIT_DEV, "temb"), temb]
        ropes = [_copy_to(rope, DIT_DEV, "rope"), rope]
        TP_STATE["block"] = self.index
        if TP_THREADS["on"] and TP_THREADS["warm"]:
            shared = [None] * len(DEVICES)
            hiddens = _run_on_both(_half_block, [(self, hiddens[s], tembs[s], ropes[s], segments, shared, LONG_VIDEO["on"])
                                                 for s in range(len(DEVICES))])
            _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
            return hiddens[1]
        if LONG_VIDEO["on"]:
            return self._forward_sliced(hiddens, tembs, ropes, segments)
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            shard = self.shards[s]
            with _on(dev):
                mod = shard.adaln_proj(tembs[s])
                mods.append(mod)
                h = _norm_modulate(shard.norm1, hiddens[s], [mod[0], mod[1]], segments)
                partials.append(shard.attn([h], rope=ropes[s]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            shard = self.shards[s]
            with _on(dev):
                h = _norm_modulate(shard.norm2, hiddens[s], [mods[s][3], mods[s][4]], segments)
                partials.append(shard.mlp([h]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]

    def _forward_sliced(self, hiddens, tembs, ropes, segments):
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            with _on(dev):
                mods.append(self.shards[s].adaln_proj(tembs[s]))
                partials.append(_sliced_attention(self.shards[s], hiddens[s], mods[s], segments, ropes[s]))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials.append(_sliced_mlp(self.shards[s], hiddens[s], mods[s], segments))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]


def split_transformer(model, progress=None):
    """Every DiT block becomes a TPBlock; the rest of the transformer (embeddings, final layer) moves to GPU 1."""
    for index in range(len(model.blocks)):
        model.blocks[index] = TPBlock(model.blocks[index], index)
        if progress is not None:
            progress(index + 1)
    for name, child in model.named_children():
        if name not in ("blocks", "token_refiner"):
            child.to(MAIN_DEV)
    for name, buffer in list(model.named_buffers(recurse=False)):
        model._buffers[name] = buffer.to(MAIN_DEV)
# <<< section


# ==== pDMD few-step LoRA: converted to Wan2GP's layer names before the split ====
# pDMD (rank 128 on every attention and FFN projection) lets H3-Image finish an image in 2 or 4 steps instead of 6. It
# runs live on top of the INT8 weights (never merged into them: re-rounding INT8 weights would erase its small deltas).
# Wan2GP's LoRA converter needs the transformer's original module layout, so the file is converted here, one block at
# a time, and attached to the tensor-parallel halves once they exist.
# >>> section:lora_prep
_LORA_PREFIXES = ("model.diffusion_model.", "diffusion_model.", "base_model.model.", "transformer.", "model.")
_LORA_SUFFIXES = {".default.weight": ".weight", ".lora.A.weight": ".lora_A.weight", ".lora.B.weight": ".lora_B.weight",
                  ".lora.down.weight": ".lora_down.weight", ".lora.up.weight": ".lora_up.weight"}


def _lora_segments(A, B):
    """[(A_i, B_i, out_start, out_end)]. Separate q / k / v adapters arrive fused as cat(A) and block_diag(B), which is
    2/3 zeros: they are split back into their three blocks (a third of the memory and the work)."""
    r3, o3 = A.shape[0], B.shape[0]
    if r3 % 3 == 0 and o3 % 3 == 0 and r3 >= 3:
        r, o = r3 // 3, o3 // 3
        off = [B[i * o:(i + 1) * o, j * r:(j + 1) * r] for i in range(3) for j in range(3) if i != j]
        if all(not bool(t.any()) for t in off):
            return [(A[i * r:(i + 1) * r].clone(), B[i * o:(i + 1) * o, i * r:(i + 1) * r].clone(), i * o, (i + 1) * o) for i in range(3)]
    return [(A, B, 0, o3)]


def prepare_lora(model, model_type, path):
    """{layer name: {"segments": [(A, B, out_start, out_end)], "alpha": a, "in": in_features, "out": out_features}} in
    Wan2GP's names (blocks.N.attn.qkv_proj, ...), read one block at a time."""
    from safetensors import safe_open

    modules = dict(model.named_modules())
    groups_out = {}
    with safe_open(path, framework="pt", device="cpu") as f:
        batches = {}
        for k in f.keys():
            m = re.search(r"blocks[._](\d+)[._]", k)
            batches.setdefault(("refiner" if "refiner" in k else "block", int(m.group(1))) if m else ("other", 0), []).append(k)
        for keys in batches.values():
            raw = {}
            for k in keys:
                key = k
                for prefix in _LORA_PREFIXES:
                    if key.startswith(prefix):
                        key = key[len(prefix):]
                        break
                for src, dst in _LORA_SUFFIXES.items():
                    if key.endswith(src):
                        key = key[:-len(src)] + dst
                        break
                raw[key] = f.get_tensor(k)
            groups = {}
            for k, v in model.preprocess_loras(model_type, raw).items():
                for down, up in (("lora_A.weight", "lora_B.weight"), ("lora_down.weight", "lora_up.weight")):
                    if k.endswith("." + down):
                        groups.setdefault(k[:-len(down) - 1], {})["A"] = v
                    elif k.endswith("." + up):
                        groups.setdefault(k[:-len(up) - 1], {})["B"] = v
                if k.endswith(".alpha"):
                    groups.setdefault(k[:-len(".alpha")], {})["alpha"] = float(v.item() if torch.is_tensor(v) else v)
            del raw
            for name, g in groups.items():
                module = modules.get(name)
                A, B = g.get("A"), g.get("B")
                if module is None or A is None or B is None:
                    continue
                in_f, out_f = module.in_features, module.out_features
                if A.shape[1] != in_f or B.shape[0] != out_f or A.shape[0] != B.shape[1]:
                    continue
                alpha = g["alpha"] / A.shape[0] if "alpha" in g else 1.0
                groups_out[name] = {"segments": _lora_segments(A.to(torch.bfloat16), B.to(torch.bfloat16)), "alpha": alpha}
    if not groups_out:
        raise ValueError(f"No layer of {os.path.basename(path)} matches MiniMax H3-Image")
    return groups_out
# <<< section


PDMD_FILE = os.path.join(MODELS_DIR, "loras", "minimax_h3_pdmd_2nfe_bf16.safetensors")
BAR.start("lora", "Loading: preparing the pDMD LoRA")
PDMD_GROUPS = prepare_lora(transformer, MODEL_TYPE, PDMD_FILE)
BAR.finish()



NUM_BLOCKS = len(transformer.blocks)
BAR.start("split", "Loading: splitting the DiT", NUM_BLOCKS, "blocks")
split_transformer(transformer, BAR.update)
BAR.finish()
gc.collect()

# ==== Exact-size pinned RAM for the token refiner ====
# The 1.4 GiB token refiner runs once per prompt: keep it in page-locked RAM (cudaHostRegister: exactly the bytes asked
# for, unlike PyTorch's power-of-two pinned blocks) and swap it onto GPU 1 only for that.
_HOST_REGISTERED = []  # registered RAM stays alive (and pinned) for the whole session
PINNED_BYTES = {}


def _dont_fork(ptr, nbytes):
    """madvise(MADV_DONTFORK): a fork() must not duplicate pinned RAM (Linux copies pinned pages into the child at once)."""
    try:
        import ctypes
        page = os.sysconf("SC_PAGE_SIZE")
        start = -(-ptr // page) * page
        end = (ptr + nbytes) // page * page
        if end > start:
            ctypes.CDLL("libc.so.6").madvise(ctypes.c_void_p(start), ctypes.c_size_t(end - start), 10)  # MADV_DONTFORK
    except Exception:
        pass


def pinned_empty_exact(nbytes):
    t = torch.empty(nbytes, dtype=torch.uint8)
    torch.cuda.check_error(torch.cuda.cudart().cudaHostRegister(t.data_ptr(), nbytes, 0))
    _dont_fork(t.data_ptr(), nbytes)
    _HOST_REGISTERED.append(t)
    return t


_refiner = transformer.token_refiner
_ref_list = list(_refiner.parameters())
_ref_offsets, _ref_total = [], 0
for _p in _ref_list:
    _ref_offsets.append(_ref_total)
    _ref_total += -(-_p.numel() * _p.element_size() // 64) * 64
try:
    _ref_buf = pinned_empty_exact(_ref_total)
    _refiner_params = []
    for _p, _off in zip(_ref_list, _ref_offsets):
        _cpu = _ref_buf[_off:_off + _p.numel() * _p.element_size()].view(_p.dtype).view(_p.shape)
        _cpu.copy_(_p.data)
        _refiner_params.append((_p, _cpu))
except Exception as e:  # no cudaHostRegister: PyTorch's (rounded) pinned blocks
    print(f"⚠️ Exact pinning unavailable ({e}); using PyTorch pinned memory")
    _refiner_params = [(p, p.data.to("cpu").pin_memory()) for p in _ref_list]
PINNED_BYTES["token refiner"] = _ref_total
for _p, _cpu in _refiner_params:
    _p.data = _cpu
_orig_preprocess_text = transformer.preprocess_text_embeds


def _preprocess_text_embeds(text_states):
    if text_states.shape[-1] == transformer.hidden_size:
        return text_states
    for p, cpu in _refiner_params:
        p.data = cpu.to(MAIN_DEV, non_blocking=True)
    try:
        return _orig_preprocess_text(text_states.to(MAIN_DEV))
    finally:
        for p, cpu in _refiner_params:
            p.data = cpu




# ==== pDMD LoRA on the tensor-parallel halves ====
# Each converted layer maps to its two halves: row-split layers (QKV, FFN-up) take the matching rows of B, column-split
# layers (output, FFN-down) the matching columns of A (their partial outputs are summed across the GPUs anyway). The
# factors stay in FP16 on the GPUs (~0.7 GB each) and the LoRA math runs on the FP16 tensor cores: a rank-128 LoRA on
# every projection adds ~3 % to a step.
# >>> section:lora
LORA_ROWS = 8192


class LoraRuntime:
    def __init__(self, model):
        self.targets = {}  # name -> [(module, device, kind, index)]
        for name, module in model.token_refiner.named_modules():
            if isinstance(module, torch.nn.Linear):
                self.targets["token_refiner." + name] = [(module, MAIN_DEV, None, None)]
        for i, block in enumerate(model.blocks):
            for short, attr, kind in (("qkv", "attn.qkv_proj", "rows"), ("out", "attn.out_proj", "cols"),
                                      ("fc1", "mlp.fc1", "rows"), ("fc2", "mlp.fc2", "cols")):
                self.targets[f"blocks.{i}.{attr}"] = [(functools.reduce(getattr, attr.split("."), block.shards[s]), DEVICES[s],
                                                       kind, block.splits[s][short]) for s in range(len(DEVICES))]
        self.adapters = {}  # id(module) -> (A, [(B, out_start, out_end, rank_start, rank_end)], alpha)
        self.handles = []
        self.strength = 0.0
        self.bytes = {}

    def load(self, groups):
        layers = 0
        for name, g in groups.items():
            parts = self.targets.get(name)
            if parts is None:
                continue
            for module, dev, kind, index in parts:
                segs = []
                for a, b, o0, o1 in g["segments"]:
                    if kind == "rows":  # this half computes the output rows `index` (in order)
                        pos = ((index >= o0) & (index < o1)).nonzero().flatten()
                        if not len(pos):
                            continue
                        assert int(pos[-1] - pos[0]) + 1 == len(pos), "non-contiguous LoRA rows in a shard"
                        b, o0, o1 = b.index_select(0, index[pos] - o0), int(pos[0]), int(pos[-1]) + 1
                    elif kind == "cols":
                        a = a[:, index]
                    segs.append((a, b, o0, o1))
                if not segs:
                    continue
                a_cat = torch.cat([a for a, _, _, _ in segs]).to(dev, torch.float16).contiguous()
                outs, r0 = [], 0
                for a, b, o0, o1 in segs:
                    outs.append((b.to(dev, torch.float16).contiguous(), o0, o1, r0, r0 + a.shape[0]))
                    r0 += a.shape[0]
                self.adapters[id(module)] = (a_cat, outs, g["alpha"])
                self.bytes[dev] = self.bytes.get(dev, 0) + a_cat.numel() * 2 + sum(b.numel() * 2 for b, *_ in outs)
                self.handles.append(module.register_forward_hook(self._hook))
            layers += 1
        return layers

    def _hook(self, module, args, output):
        if self.strength == 0.0:
            return output
        a16, outs, alpha = self.adapters[id(module)]
        s = self.strength * alpha
        x = args[0]
        x2 = x.reshape(-1, x.shape[-1])
        out2 = output.view(-1, output.shape[-1])
        # The input is scaled by a power of two before the FP16 cast and the result scaled back, so large activations
        # (FFN-down input) cannot overflow; FP16's 11-bit mantissa is plenty for a correction term.
        pre = 1.0 / 512 if x2.shape[-1] > 8192 else 1.0 / 64
        for start in range(0, x2.shape[0], LORA_ROWS):
            end = min(x2.shape[0], start + LORA_ROWS)
            h = (x2[start:end] * pre).to(torch.float16) @ a16.t()
            for b16, o0, o1, r0, r1 in outs:
                out2[start:end, o0:o1].add_((h[:, r0:r1] @ b16.t()).to(out2.dtype), alpha=s / pre)
            del h
        return output
# <<< section


BAR.start("lora", "Loading: pDMD LoRA onto both GPUs")
lora_runtime = LoraRuntime(transformer)
_lora_layers = lora_runtime.load(PDMD_GROUPS)
del PDMD_GROUPS
gc.collect()
BAR.finish()
if _lora_layers < 200:  # 50 blocks x 4 projections + the token refiner
    print(f"⚠️ The pDMD LoRA matched only {_lora_layers} layers")

transformer.preprocess_text_embeds = _preprocess_text_embeds

# ==== Text encoder + image VAE: MMGP on GPU 1, unloaded before denoising ====
aux_pipe = {
    "text_encoder": pipeline.text_encoder.language_model,
    "vision_encoder": pipeline.text_encoder.visual,
    "vae": pipeline.video_decoder,
    "video_encoder": pipeline.video_encoder,
}
if DISTILLED_TE:
    aux_pipe["text_connector"] = pipeline.text_encoder.text_connector
pipeline.video_decoder._budget = 0  # decode with the whole VAE resident
if TE_DTYPE == torch.float32:
    pipeline.text_encoder.language_model._convertWeightsFloatTo = None
    pipeline.text_encoder.visual._convertWeightsFloatTo = None
# The text encoder and the image VAE are pinned in RAM: Kaggle's disk reads at ~180 MB/s, so a memory-mapped copy
# that dropped out of the page cache would cost seconds per image before anything could start.
_pin = ["text_encoder", "vae", "video_encoder"] + (["text_connector"] if DISTILLED_TE else [])
_profile_args = dict(profile_no=4, quantizeTransformer=False, extraModelsToQuantize=None, convertWeightsFloatTo=torch.bfloat16,
                     asyncTransfers=True, verboseLevel=-1, budgets={"text_encoder": 1500, "vision_encoder": 0, "*": 3000})
BAR.start("pin", "Loading: pinning RAM")
try:
    aux_offload = offload.profile(aux_pipe, pinnedMemory=_pin, **_profile_args)
except Exception as e:
    print(f"⚠️ Pinning failed ({e}); the text encoder streams from regular RAM")
    aux_offload = offload.profile(aux_pipe, pinnedMemory=False, **_profile_args)
pipeline.set_offload_handoff(aux_offload, None)  # the pipeline unloads these before handing the GPU to the DiT
BAR.finish()
if hasattr(torch._C, "_host_emptyCache"):
    torch._C._host_emptyCache()

gc.collect()
for _dev in DEVICES:
    with torch.cuda.device(_dev):
        torch.cuda.empty_cache()
_free0, _free1 = torch.cuda.mem_get_info(0)[0], torch.cuda.mem_get_info(1)[0]


# ==== Attention: exact FP16 SDPA, or Comfy Kitchen's INT8 attention ====
# Attention is the largest part of a denoising step at 16 MP (~16k image tokens).
# Fast (default): Comfy Kitchen's INT8 attention, a compiled CUDA kernel with Turing (SM 7.5) code. Q and K get a Hadamard
# rotation (same dot products, smaller outliers) and go to INT8, V to INT8 per channel, the softmax weights to unsigned
# INT8, with the softmax itself in FP32; the T4's INT8 tensor cores run at twice the FP16 rate. It is offered only when a
# startup test on both GPUs passes and shows it faster, and every INT8 run still compares it with exact attention on the
# user's image in
# its first step (blocks 1 and 26, GPU 0), switching back to exact by itself if they differ too much.
# Exact: PyTorch SDPA in FP16 (its memory-efficient kernel; the T4 has no flash kernel).
# Wan2GP's own SageAttention path is Triton INT8 code, which Kaggle's Triton cannot compile for the T4.
# >>> section:attention
ATTENTION_EXACT, ATTENTION_INT8 = "Exact (FP16)", "Fast (Kitchen INT8)"
ATTENTION_MAX_DIFF = 0.08  # relative L2 difference from exact attention above which an INT8 run switches back to exact
INT8_MIN_TOKENS = 2048     # INT8 only for the DiT's long sequences (the text token refiner stays exact)
INT8_DEVICE_TYPES = ("cuda",)
ATTENTION = {"mode": "sdpa", "int8": None, "int8_dtype": None, "int8_contiguous": False, "int8_speedup": None, "int8_reason": "",
             "checks": set(), "fallback": None}
offload.shared_state["_attention"] = "sdpa"
import shared.attention as _w2g_attention
_sdpa_attention = _w2g_attention.pay_attention  # Wan2GP's own (SDPA here), never the wrapper below
_ATTENTION_CHECK_LOCK = threading.Lock()


def _int8_attention(q, k, v):
    """Kitchen INT8 attention on [batch, tokens, heads, dim] tensors (H3's layout), output in q's dtype and layout."""
    out_dtype, dtype = q.dtype, ATTENTION["int8_dtype"] or q.dtype
    args = [t.transpose(1, 2) if t.dtype == dtype else t.transpose(1, 2).to(dtype) for t in (q, k, v)]
    if ATTENTION["int8_contiguous"]:
        args = [t.contiguous() for t in args]
    return ATTENTION["int8"].int8_attention(*args).transpose(1, 2).to(out_dtype)


def _relative_diff(out, ref):
    """||out - ref|| / ||ref|| in FP32, by slices of tokens (no full-size FP32 copies)."""
    num = torch.zeros((), dtype=torch.float32, device=out.device)
    den = torch.zeros((), dtype=torch.float32, device=out.device)
    for a in range(0, out.shape[1], 4096):
        o, r = out[:, a:a + 4096].float(), ref[:, a:a + 4096].float()
        num += (o - r).square().sum()
        den += r.square().sum()
    return (num / den.clamp_min(1e-30)).sqrt().item()


def _check_int8_attention(q, k, v):
    """INT8 vs exact attention on this block's real Q / K / V. Returns the output to use (exact if INT8 is too far off)."""
    dev = q.device
    if dev.type == "cuda":
        free = torch.cuda.mem_get_info(dev)[0] + torch.cuda.memory_reserved(dev) - torch.cuda.memory_allocated(dev)
        if free < 8 * q.numel() * q.element_size() + (512 << 20):  # the exact reference needs copies of Q / K / V
            return _int8_attention(q, k, v)
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    out = _int8_attention(q, k, v)
    diff = _relative_diff(out, ref)
    if not math.isfinite(diff) or diff > ATTENTION_MAX_DIFF:
        ATTENTION.update(mode="sdpa", fallback=diff)
        return ref
    return out


def pay_attention_tp(qkv_list, recycle_q=False, **kwargs):
    """H3's attention call: Kitchen INT8 for the DiT's long sequences when the run asks for it, else Wan2GP's (SDPA)."""
    q = qkv_list[0]
    if (ATTENTION["mode"] != "int8" or kwargs or q.dim() != 4 or q.shape[1] < INT8_MIN_TOKENS
            or q.device.type not in INT8_DEVICE_TYPES):
        return _sdpa_attention(qkv_list, recycle_q=recycle_q, **kwargs)
    q, k, v = qkv_list
    qkv_list.clear()
    check = False
    if q.device == DEVICES[0] and ATTENTION["checks"]:
        with _ATTENTION_CHECK_LOCK:  # one check per listed block, whichever thread gets there first
            check = TP_STATE["block"] in ATTENTION["checks"]
            ATTENTION["checks"].discard(TP_STATE["block"])
    return _check_int8_attention(q, k, v) if check else _int8_attention(q, k, v)


_h3_transformer.pay_attention = pay_attention_tp


def _probe_case(dev, fused):
    """INT8 vs SDPA difference on random BF16 data in one of H3's two layouts: fused (Q and K are views of one fused
    projection output, V a copy: the normal path) or separate contiguous tensors (long-sequence mode)."""
    g = torch.Generator(device=dev).manual_seed(0)
    tokens, heads, dim = 3000, 4, 128
    if fused:
        qkv = torch.randn(1, tokens, 3 * heads * dim, device=dev, generator=g).to(torch.bfloat16)
        q, k, v = (t.view(1, tokens, heads, dim) for t in qkv.split(heads * dim, dim=-1))
        v = v.clone()
    else:
        q, k, v = (torch.randn(1, tokens, heads, dim, device=dev, generator=g).to(torch.bfloat16) for _ in range(3))
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    return _relative_diff(_int8_attention(q, k, v), ref)


def _probe_int8_attention():
    """Kitchen INT8 attention on both GPUs against SDPA on random data (catches a kernel that does not load, a dtype it
    does not take on Turing, or strides it does not follow), then a speed test at a DiT-like size."""
    try:
        import comfy_kitchen
        if not all(comfy_kitchen.int8_attention_is_available(dev) for dev in DEVICES):
            ATTENTION["int8_reason"] = "not available on this GPU"
            return
        ATTENTION["int8"] = comfy_kitchen
        # BF16 straight from the projections and the projections' own strides first; FP16 / contiguous copies otherwise
        for dtype, contiguous in ((torch.bfloat16, False), (torch.bfloat16, True), (torch.float16, False), (torch.float16, True)):
            ATTENTION.update(int8_dtype=dtype, int8_contiguous=contiguous)
            try:
                worst = 0.0
                for dev in DEVICES:
                    with _on(dev), torch.inference_mode():
                        worst = max(worst, _probe_case(dev, True), _probe_case(dev, False))
                if math.isfinite(worst) and worst < 0.1:
                    break
                ATTENTION["int8_reason"] = f"self-test differed {worst:.1%} from SDPA"
            except Exception as e:
                ATTENTION["int8_reason"] = f"self-test failed: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}"
            ATTENTION["int8_dtype"] = None
        if ATTENTION["int8_dtype"] is None:
            ATTENTION["int8"] = None
            return
        ATTENTION["int8_reason"] = ""
        with _on(MAIN_DEV), torch.inference_mode():
            q, k, v = (torch.randn(1, 8192, 28, 128, device=MAIN_DEV).to(torch.bfloat16) for _ in range(3))
            timings = []
            for run in (lambda: _sdpa_attention([q, k, v]), lambda: _int8_attention(q, k, v)):
                for _ in range(2):
                    run()
                start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(3):
                    run()
                end.record()
                end.synchronize()
                timings.append(start.elapsed_time(end))
            ratio = timings[0] / max(timings[1], 1e-3)
            del q, k, v
        if ratio < 1.05:  # offered only when it is actually faster on these GPUs
            ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"not faster than exact attention here: {ratio:.2f}x")
        else:
            ATTENTION["int8_speedup"] = ratio
    except Exception as e:
        ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"unavailable: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}")
    finally:
        for dev in DEVICES:
            with _on(dev):
                torch.cuda.empty_cache()


def attention_speedup(choice):
    return (ATTENTION["int8_speedup"] or 1.0) if choice == ATTENTION_INT8 and ATTENTION["int8"] is not None else 1.0
# <<< section


BAR.start("attention", "Loading: testing Kitchen INT8 attention", 1, "test")
_probe_int8_attention()
BAR.finish()
ATTENTION_CHOICES = ([ATTENTION_INT8] if ATTENTION["int8"] is not None else []) + [ATTENTION_EXACT]
ATTENTION_DEFAULT = ATTENTION_CHOICES[0]
ATTENTION_LABEL = "Kitchen INT8 attention" if ATTENTION["int8"] is not None else "SDPA attention"

# ==== VAE decode in groups of tiles ====
# Wan2GP decodes an image as 256 px tiles stacked into ONE decoder batch (several hundred tiles at 16 MP). The decoder
# treats every tile on its own, so running the tiles in smaller groups gives exactly the same pixels. The group size
# is planned from GPU 1's free VRAM before each decode; halving on an out-of-memory error stays as a last resort.
VAE_TILE_GROUP = {}
VAE_TILE_BYTES = 40 * 2**20


def _tile_grouped(decoder):
    full_forward = decoder.forward

    def forward(hidden_states, *args, **kwargs):
        dev, tiles = hidden_states.device.index, hidden_states.shape[0]
        while True:
            n = min(VAE_TILE_GROUP.get(dev, tiles), tiles)
            try:
                if n >= tiles:
                    return full_forward(hidden_states, *args, **kwargs)
                return torch.cat([full_forward(hidden_states[i:i + n], *args, **kwargs) for i in range(0, tiles, n)])
            except torch.OutOfMemoryError:
                if n <= 1:
                    raise
                VAE_TILE_GROUP[dev] = max(1, n // 2)
                print(f"  ⚙️ VAE: decoding {VAE_TILE_GROUP[dev]} tiles at a time from now on (out of memory with {n})")
                with torch.cuda.device(hidden_states.device):
                    torch.cuda.synchronize()  # kernels queued before the failed allocation may still use freed blocks
                    gc.collect()
                    torch.cuda.empty_cache()

    decoder.forward = forward


_tile_grouped(pipeline.vae.decoder)


def _plan_tile_groups():
    """Tiles per decoder call that fit GPU 1's free VRAM right now (it loads the decoder on first use)."""
    resident = any(p.is_cuda for p in pipeline.video_decoder.parameters())
    decoder_bytes = sum(p.numel() * p.element_size() for p in pipeline.video_decoder.parameters())
    with torch.cuda.device(MAIN_DEV):
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info(MAIN_DEV)[0] - (0 if resident else decoder_bytes)
    VAE_TILE_GROUP[MAIN_DEV.index] = max(1, int((free - 768 * 2**20) // VAE_TILE_BYTES))


PEAKS = {}
_orig_vae_decode = pipeline.vae.decode


def _vae_decode(*args, **kwargs):
    PEAKS["denoise"] = [torch.cuda.max_memory_allocated(d) for d in DEVICES]  # the DiT phase ends here
    for d in DEVICES:
        torch.cuda.reset_peak_memory_stats(d)
    report(0.92, "🎞️ VAE decoding")
    _plan_tile_groups()
    return _orig_vae_decode(*args, **kwargs)


pipeline.vae.decode = _vae_decode

# ==== Drop the checkpoint files from the page cache ====
# Loading reads ~40 GB of checkpoints and Linux keeps what fits of them cached. That cache is reclaimable, but Kaggle's
# RAM meter counts it. Everything is pinned or on the GPUs now, so the cached copies are useless.
def drop_file_cache(*roots):
    for root in roots:
        for dirpath, _, files in os.walk(root, followlinks=True):
            for name in files:
                try:
                    fd = os.open(os.path.join(dirpath, name), os.O_RDONLY)
                    try:
                        os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                    finally:
                        os.close(fd)
                except (OSError, AttributeError):
                    pass


def _ram_report():
    vm = psutil.virtual_memory()
    cache = getattr(vm, "cached", 0) + getattr(vm, "buffers", 0)
    return f"{_ram_in_use()} used by the engine | {_gb(cache)} file cache (reclaimable, Kaggle's meter counts it)"


drop_file_cache(MODELS_DIR)

# ==== Sizes, time estimates ====
# MiniMax H3-Image works best at about 16 MP; 4 MP is the smallest canvas it generates (smaller sizes save no time).
SIZES = {
    "2048x2048 · 1:1 · 4 MP": (2048, 2048), "2368x1760 · 4:3 · 4 MP": (2368, 1760), "1760x2368 · 3:4 · 4 MP": (1760, 2368),
    "2496x1664 · 3:2 · 4 MP": (2496, 1664), "1664x2496 · 2:3 · 4 MP": (1664, 2496), "2720x1536 · 16:9 · 4 MP": (2720, 1536),
    "1536x2720 · 9:16 · 4 MP": (1536, 2720),
    "4096x4096 · 1:1 · 16 MP": (4096, 4096), "4736x3552 · 4:3 · 16 MP": (4736, 3552), "3552x4736 · 3:4 · 16 MP": (3552, 4736),
    "5024x3360 · 3:2 · 16 MP": (5024, 3360), "3360x5024 · 2:3 · 16 MP": (3360, 5024), "5472x3072 · 16:9 · 16 MP": (5472, 3072),
    "3072x5472 · 9:16 · 16 MP": (3072, 5472), "6272x2688 · 21:9 · 16 MP": (6272, 2688), "2688x6272 · 9:21 · 16 MP": (2688, 6272),
}
SIZE_OF = {v: k for k, v in SIZES.items()}
DEFAULT_SIZE = "2048x2048 · 1:1 · 4 MP"
MAX_IMAGES = 4
EDIT_OWN, EDIT_SIZE = "Keep Picture 1's proportions", "Use the selected size"
# name -> (steps, pDMD LoRA strength). 6 steps is the model's own schedule (one output head per step); 2 and 4 steps run
# the pDMD LoRA on H3's shift-12 grid, the one it was trained on.
SPEEDS = {"⚡ Turbo: 2 steps": (2, 1.0), "⚖️ Balanced: 4 steps": (4, 1.0), "💎 Quality: 6 steps (official)": (6, 0.0)}
SPEED_TURBO, SPEED_QUALITY = "⚡ Turbo: 2 steps", "💎 Quality: 6 steps (official)"
DEFAULT_SPEED = SPEED_TURBO


def fit_area(width, height, area):
    """A size of about `area` pixels with the proportions of width x height, sides multiples of 32."""
    ratio = width / height
    return max(32, round(math.sqrt(area * ratio) / 32) * 32), max(32, round(math.sqrt(area / ratio) / 32) * 32)


def output_size(size_label, images, edit_mode):
    w, h = SIZES.get(size_label, SIZES[DEFAULT_SIZE])
    if images and edit_mode == EDIT_OWN:
        w, h = fit_area(*images[0].size, w * h)
    return w, h


def image_rows(w, h, images):
    """Packed sequence rows of one forward: prompt (+ what the text encoder sees of each picture), the target canvas
    and every picture's latent (each token covers 32 x 32 pixels)."""
    cw, ch = canvas_size(w, h)
    rows = 700 + (cw // 32) * (ch // 32)
    for image in images:
        iw, ih = bound_short_edge(*image.size, *H3_IMAGE_REFERENCE_SHORT_EDGE)
        rows += (iw // 32) * (ih // 32) + 1400
    return rows


def step_seconds(rows, attention_speedup=1.0):
    """Seconds per step on Kaggle T4 x2 with exact attention: 36 x^2 + 6 x + 2, x = rows / 16k (2K: 5.3 s with Kitchen
    INT8 attention, 16 MP: 26 s). The quadratic term is attention, divided by the speedup of the chosen attention."""
    x = rows / 16384
    return 36 * x * x / min(attention_speedup, 2.0) + 6 * x + 2


def needed_bytes(rows, long_video=False):
    """Working VRAM per GPU: activations per packed row (half the heads per GPU) + scratch."""
    return rows * (48 if long_video else 110) * 1024 + 700 * 1024**2


def total_seconds(w, h, images, attention, speed=DEFAULT_SPEED):
    steps, lora = SPEEDS.get(speed, SPEEDS[DEFAULT_SPEED])
    rows = image_rows(w, h, images)
    cw, ch = canvas_size(w, h)
    prompt = 2 if TEXT_ENCODER == "fast" else 12
    decode_and_save = 2.6 * cw * ch / 4.2e6
    return 1 + prompt + 6 * len(images) + steps * step_seconds(rows, attention_speedup(attention)) * (1.05 if lora else 1.0) + decode_and_save


# ==== Live progress (also polled by the page over plain HTTP, which survives tunnels dropping Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "video": None, "message": "", "final_n": 0}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def free_for_denoise():
    aux_offload.unload_all()
    gc.collect()
    frees = []
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
        frees.append(torch.cuda.mem_get_info(dev)[0])
    return frees


def save_png(x, path):
    """[3, 1, H, W] (uint8, or float in [-1, 1]) to a PNG."""
    frame = x[:, 0] if x.dim() == 4 else x
    if frame.dtype != torch.uint8:
        frame = frame.float().clamp(-1, 1).add(1).mul(127.5).round().to(torch.uint8)
    Image.fromarray(frame.permute(1, 2, 0).contiguous().cpu().numpy()).save(path, compress_level=4)


# ==== Encoded pictures are cached between images ====
# Re-running an edit with the same pictures (another seed, a tweaked prompt) reuses their VAE latents (the encode is
# deterministic: same input, same latents). The prompt encoding has its own cache in Wan2GP.
import collections
import hashlib

ENCODE_CACHE = collections.OrderedDict()


def attention_warning(int8):
    """A status line only when an INT8 run had to switch back to exact attention."""
    if not int8 or ATTENTION["fallback"] is None:
        return ""
    return "⚠️ Fast attention was not accurate enough on this image, so it was finished with exact attention"


def install_hooks(pipe):
    """The encode cache and the switch to per-GPU threads after the first transformer pass, on this pipeline."""
    orig_encode_video = pipe._encode_video
    orig_forward = pipe.transformer.forward

    def cached_encode_video(video, keep_all_latents=False):
        if video.is_cuda:
            return orig_encode_video(video, keep_all_latents=keep_all_latents)
        data = video.contiguous()
        digest = hashlib.blake2b(memoryview(data.view(torch.uint8).reshape(-1).numpy()), digest_size=16).hexdigest()
        vae = pipe.vae
        key = (id(pipe), digest, str(video.dtype), tuple(video.shape), bool(keep_all_latents), getattr(vae, "use_tiling", None),
               getattr(vae, "tile_sample_min_height", None), getattr(vae, "tile_sample_min_width", None))
        if key in ENCODE_CACHE:
            ENCODE_CACHE.move_to_end(key)
            return ENCODE_CACHE[key].clone()
        latent = orig_encode_video(video, keep_all_latents=keep_all_latents)
        ENCODE_CACHE[key] = latent.clone()
        while len(ENCODE_CACHE) > 16:
            ENCODE_CACHE.popitem(last=False)
        return latent

    def forward(*args, **kwargs):
        out = orig_forward(*args, **kwargs)
        TP_THREADS["warm"] = True  # the first pass of the session (one thread) has compiled every kernel on both GPUs
        return out

    pipe._encode_video = cached_encode_video
    pipe.transformer.forward = forward


if "pipeline" in globals():
    install_hooks(pipeline)


@torch.inference_mode()
def run_generation(kwargs, steps):
    pipeline._interrupt = False
    t0 = time.time()
    timings = {"encode": 0.0, "denoise": 0.0, "decode": 0.0}
    phase = {"name": "prep", "t": time.time(), "step_t": time.time()}

    def callback(step, latent=None, is_start=False, override_num_inference_steps=None, progress_unit=None,
                 progress_title=None, **kw):
        now = time.time()
        if progress_unit == "layers":  # Qwen3-VL prompt encoding
            total = override_num_inference_steps or 50
            BAR.start("text", "Prompt encoding", total, "layers")
            BAR.update(max(0, step + 1), total)
            report(0.05 + 0.10 * min(1.0, (step + 1) / total), f"📝 Encoding the prompt {max(0, step + 1)}/{total}")
            return
        if progress_unit == "tiles":  # VAE tiles: the pictures before denoising, the image after it
            total = override_num_inference_steps or 1
            done = min(total, max(0, step + 1))
            if phase["name"] == "decode":
                BAR.start("decode", "VAE decoding", total, "tiles")
                BAR.update(done, total)
                report(0.92 + 0.06 * done / total, f"🎞️ VAE decoding {done}/{total}")
            else:
                BAR.start("encode", "Encoding pictures", total, "tiles")
                BAR.update(done, total)
                report(0.02 + 0.03 * done / total, f"🖼️ Encoding the pictures {done}/{total}")
            return
        if is_start:
            timings["encode"] = now - t0
            phase.update(name="denoise", t=now, step_t=now, total=override_num_inference_steps or steps)
            BAR.start("denoise", "Denoising", phase["total"], "steps")
            report(0.15, f"🎨 Denoising step 0/{phase['total']}")
            return
        total = phase.get("total", steps)
        dt = now - phase["step_t"]
        phase["step_t"] = now
        done = step + 1
        BAR.update(done, total)
        report(0.15 + 0.75 * done / total, f"🎨 Denoising step {done}/{total} ({dt:.0f}s/step)")
        if done >= total:
            timings["denoise"] = now - phase["t"]
            phase.update(name="decode", t=now)

    report(0.01, "⚡ Preparing...")
    BAR.start("prep", "Preparing inputs")
    try:
        result = pipeline.generate(callback=callback, **kwargs)
    finally:
        BAR.finish()
    if result is None:
        return None, timings
    if phase["name"] == "decode":
        timings["decode"] = time.time() - phase["t"]
    elif phase["name"] == "denoise":
        timings["denoise"] = time.time() - phase["t"]
    timings["total"] = time.time() - t0
    return result["x"], timings


def on_main_gpu(fn):
    """CUDA's current device is per thread: Gradio workers start on GPU 0. Run handlers with GPU 1 current, the device
    the sampler, text encoder and VAE use for "cuda"."""
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        with torch.cuda.device(MAIN_DEV):
            return fn(*args, **kwargs)
    return wrapped


def _pil(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = Image.open(value)
    return value.convert("RGB")


@on_main_gpu
def generate_image(prompt, size_label, edit_mode, img1, img2, img3, img4, seed, attention=ATTENTION_EXACT, speed=DEFAULT_SPEED):
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    try:
        prompt = (prompt or "").strip()
        if not prompt:
            return None, "❌ Write a prompt first: describe the whole image in detail."
        images = [_pil(img) for img in (img1, img2, img3, img4) if img is not None]
        w, h = output_size(size_label, images, edit_mode)
        if seed is None or int(seed) < 0:
            seed = random.randint(0, 2**31 - 1)
        seed = int(seed)

        rows = image_rows(w, h, images)
        frees = free_for_denoise()
        LONG_VIDEO["on"] = needed_bytes(rows) > min(frees)
        if needed_bytes(rows, LONG_VIDEO["on"]) > min(frees):
            return None, (f"❌ {w}x{h} with {len(images)} picture(s) needs more GPU memory than is free "
                          f"({_gb(frees[0])} / {_gb(frees[1])}). Use fewer or smaller pictures, or a 4 MP size.")
        int8 = attention == ATTENTION_INT8 and ATTENTION["int8"] is not None
        steps, lora_strength = SPEEDS.get(speed, SPEEDS[DEFAULT_SPEED])
        estimate = total_seconds(w, h, images, attention, speed)
        lora_runtime.strength = lora_strength
        kwargs = dict(input_prompt=prompt, frame_num=1, height=h, width=w, image_mode=1, sampling_steps=steps,
                      shift=12.0, seed=seed, sample_solver="euler", guide_phases=1, batch_size=1,
                      input_ref_images=images or None, image_refs_relative_size=100,
                      video_prompt_type=("KI" if edit_mode == EDIT_OWN else "I") if images else "")
        what = f"edit with {len(images)} picture(s)" if images else "text to image"
        print(f"🎨 {w}x{h} | {what} | {steps} steps{' | fast attention' if int8 else ''} | seed {seed} | est. ~{_fmt_secs(estimate)}")
        sys.stdout.flush()

        for dev in DEVICES:
            torch.cuda.reset_peak_memory_stats(dev)
        PEAKS.clear()
        # INT8 attention is checked against exact attention on this image in the first step (blocks 1 and 26, GPU 0)
        ATTENTION.update(mode="int8" if int8 else "sdpa", checks={0, NUM_BLOCKS // 2} if int8 else set(), fallback=None)
        result, timings = run_generation(kwargs, steps)
        if result is None:
            return None, "⏹️ Generation stopped."
        report(0.98, "💾 Saving PNG...")
        out_path = os.path.join(OUTPUT_DIR, f"h3_image_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.png")
        BAR.start("save", "Saving PNG")
        save_png(result, out_path)
        BAR.finish()
        out_h, out_w = result.shape[-2:]
        total = timings.get("total", 0.0)
        per_step = timings["denoise"] / steps
        status = "\n".join([
            f"✅ Done in {_fmt_secs(total)}  (MiniMax H3-Image, {what})",
            f"⚡ Prompt + pictures {timings['encode']:.0f}s | Denoising {timings['denoise']:.0f}s ({per_step:.1f}s/step) | VAE {timings['decode']:.0f}s",
            f"📐 {out_w}x{out_h} | 🎲 Seed {seed} | {speed} | "
            + ("⚡ Fast attention" if int8 and ATTENTION["fallback"] is None else "exact attention"),
            f"🧠 Peak VRAM per GPU: "
            + (f"denoise {_gb(PEAKS['denoise'][0])} / {_gb(PEAKS['denoise'][1])}, " if "denoise" in PEAKS else "")
            + f"decode {_gb(torch.cuda.max_memory_allocated(0))} / {_gb(torch.cuda.max_memory_allocated(1))}",
            f"📁 {out_path}",
        ] + [line for line in (attention_warning(int8),) if line])
        print(f"✅ Done in {_fmt_secs(total)} ({per_step:.0f}s/step) → {os.path.basename(out_path)}")
        drop_file_cache(OUTPUT_DIR)
        sys.stdout.flush()
        return out_path, status
    except torch.OutOfMemoryError:
        traceback.print_exc()
        return None, "❌ Out of GPU memory. Use a 4 MP size or fewer pictures and try again."
    except Exception as e:
        traceback.print_exc()
        if "CUDA error" in str(e):  # e.g. an illegal memory access: the GPU context is unusable from here on
            print("❌ The GPUs are in an error state. Re-run Step 2 to restart the engine (the models reload in ~3 min).")
            return None, f"❌ GPU error: {e}\nRe-run Step 2 to restart the engine."
        return None, f"❌ Error: {e}"
    finally:
        ATTENTION.update(mode="sdpa", checks=set())
        lora_runtime.strength = 0.0
        BAR.abort()
        PROGRESS["active"] = False
        for key in _tp_cache:  # last hidden state / rope / temb copies held on both GPUs
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


# ==== Generation as a background job ====
# The Generate button only starts a job and returns; the page script polls /aiq_progress over plain HTTP and clicks a
# hidden button to fetch the result when a new one is ready, so a reloaded page or a dropped connection loses nothing.
JOB = {"id": 0, "done": 0, "running": False, "image": None, "status": "Ready."}


def _job_worker(job_id, args):
    try:
        image, status = generate_image(*args)
    except BaseException as e:  # generate_image reports its own errors; this is a last resort
        image, status = None, f"❌ Error: {e}"
    JOB.update(running=False, done=job_id, image=image, status=status)
    PROGRESS.update(active=False, finished=time.time(), video=os.path.basename(image) if image else None,
                    message=status.splitlines()[0] if status else "", final_n=job_id)


def start_generation(*args):
    if JOB["running"]:
        return "⏳ An image is already being generated. It appears here as soon as it is ready."
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return ("🚀 Started. Progress shows at the top of the page and the image appears below when it is ready. "
            "Reloading the page or a dropped connection does not stop the job.")


def deliver_result():
    if JOB["running"]:
        return gr.skip(), f"⏳ {PROGRESS['text'] or 'Working...'}"
    if not JOB["done"]:
        return gr.skip(), JOB["status"]
    return JOB["image"], JOB["status"]


def stop_generation():
    if not JOB["running"]:
        return "Nothing is running."
    pipeline._interrupt = True
    return "⏹️ Stopping after the current step..."


def latest_image_path():
    images = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")), key=os.path.getmtime)
    return images[-1] if images else None


def load_latest_image():
    path = latest_image_path()
    if not path:
        return None, f"No images in {OUTPUT_DIR} yet."
    return path, f"📁 Loaded the latest image: {path}"


def size_info(size_label, edit_mode, img1, img2, img3, img4, attention=ATTENTION_EXACT, speed=DEFAULT_SPEED):
    images = [_pil(img) for img in (img1, img2, img3, img4) if img is not None]
    w, h = output_size(size_label, images, edit_mode)
    note = f" (Picture 1's proportions)" if images and edit_mode == EDIT_OWN else ""
    pictures = f" with {len(images)} picture(s)" if images else ""
    return f"ℹ️ Output {w}x{h}{note}{pictures}: about {_fmt_secs(total_seconds(w, h, images, attention, speed))} per image."


# ==== Official examples (MiniMax's own prompts, sizes, seeds and reference image) ====
H3_IMAGE_REVISION = "8ced60068623def5b74b4d10558117cb61d6482d"  # MiniMaxAI/MiniMax-H3-Image
EXAMPLE_FILES = ["revised_prompt.t2i.txt", "revised_prompt.edit.txt", "reference1.png"]


def ensure_examples():
    """MiniMax's example files (Step 1 downloads them; fetched here if they are missing)."""
    missing = [f for f in EXAMPLE_FILES if not os.path.exists(os.path.join(EXAMPLES_DIR, f))]
    if not missing:
        return True
    try:
        from huggingface_hub import hf_hub_download
        os.makedirs(EXAMPLES_DIR, exist_ok=True)
        for f in missing:
            path = hf_hub_download("MiniMaxAI/MiniMax-H3-Image", f"examples/{f}", revision=H3_IMAGE_REVISION,
                                   local_dir=os.path.dirname(EXAMPLES_DIR))
            if os.path.abspath(path) != os.path.abspath(os.path.join(EXAMPLES_DIR, f)):
                shutil.copy(path, os.path.join(EXAMPLES_DIR, f))
        return True
    except Exception as e:
        print(f"⚠️ The official examples could not be downloaded ({str(e).splitlines()[0][:120]}); the UI shows none")
        return False


def official_example_rows():
    """Rows for gr.Examples: [prompt, size, edit mode, picture 1-4, seed, speed]."""
    paths = {f: os.path.join(EXAMPLES_DIR, f) for f in EXAMPLE_FILES}
    if not all(os.path.exists(p) for p in paths.values()):
        return [], []

    def read(name):
        with open(paths[name], encoding="utf-8") as f:
            return f.read().strip()

    t2i, edit = read("revised_prompt.t2i.txt"), read("revised_prompt.edit.txt")
    ref = Image.open(paths["reference1.png"]).convert("RGB")
    rows = [[t2i, SIZE_OF[(2368, 1760)], EDIT_OWN, None, None, None, None, 42, SPEED_TURBO],
            [t2i, SIZE_OF[(4736, 3552)], EDIT_OWN, None, None, None, None, 42, SPEED_QUALITY],
            [edit, SIZE_OF[(4096, 4096)], EDIT_SIZE, paths["reference1.png"], None, None, None, 1234, SPEED_QUALITY]]

    def took(w, h, images, speed):
        seconds = total_seconds(w, h, images, ATTENTION_DEFAULT, speed)
        return f"~{round(seconds / 5) * 5} s" if seconds < 90 else f"~{round(seconds / 60)} min"

    labels = [f"⚡ Farm-to-table feast, 2K quick test, 2 steps ({took(2368, 1760, [], SPEED_TURBO)})",
              f"🥖 Farm-to-table feast, official 16 MP text to image, 6 steps ({took(4736, 3552, [], SPEED_QUALITY)})",
              f"🎙️ Podcast hosts fist-bump, official 16 MP edit, 6 steps ({took(4096, 4096, [ref], SPEED_QUALITY)})"]
    return rows, labels


ensure_examples()


# ==== Warm-up: one 2K step at startup ====
# The first transformer pass of a session runs on one thread and sets up every kernel on both GPUs; a 1-step 2K image
# here does that (and loads the text encoder and VAE once), so the first image you generate is as fast as the next ones.
@torch.inference_mode()
def warm_up():
    BAR.start("warm", "Loading: warm-up image")
    try:
        with torch.cuda.device(MAIN_DEV):
            ATTENTION.update(mode="int8" if ATTENTION["int8"] is not None else "sdpa", checks=set(), fallback=None)
            lora_runtime.strength = 1.0
            free_for_denoise()
            pipeline._interrupt = False
            pipeline.generate(input_prompt="The image is a photograph of a plain grey studio wall in soft daylight.", frame_num=1,
                              height=2048, width=2048, image_mode=1, sampling_steps=1, shift=12.0, seed=0, sample_solver="euler",
                              guide_phases=1, batch_size=1, video_prompt_type="", callback=None)
    except Exception as e:
        print(f"⚠️ Warm-up skipped ({str(e).splitlines()[0][:120] if str(e) else type(e).__name__}); the first image takes a little longer")
    finally:
        ATTENTION.update(mode="sdpa", checks=set(), fallback=None)
        lora_runtime.strength = 0.0
        BAR.finish()
        PROGRESS.update(active=False, fraction=0.0, text="")
        for key in _tp_cache:
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


if os.environ.get("H3I_WARMUP", "1") == "1":
    warm_up()


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#aiq-deliver { display: none !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎨 MiniMax H3-Image: 16 MP Text to Image + Editing</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
STATUS_HTML = (f'<div class="status-line">⚡ {INT8_LABEL} &nbsp;|&nbsp; {ATTENTION_LABEL} &nbsp;|&nbsp; MiniMax H3-Image 20B '
               f'tensor-parallel on 2 x T4 &nbsp;|&nbsp; pDMD 2-step turbo &nbsp;|&nbsp; Powered by Wan2GP</div>')
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved<br>'
               'Model: <a href="https://huggingface.co/MiniMaxAI/MiniMax-H3-Image" target="_blank">MiniMax H3-Image</a> '
               '(MiniMax H3-Image Community License) · label results as AI-generated.</div>')

PROGRESS_HEAD = """
<script>
(function () {
  // Progress overlay fed by plain HTTP polling: it keeps working when a tunnel drops Gradio's event stream. When a new
  // result is ready it clicks the hidden deliver button once, which fetches the image (no timer re-rendering the page).
  var box = null, fill = null, label = null, hideAt = 0, lastFinal = null;
  function build() {
    box = document.createElement("div");
    box.style.cssText = "position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:99999;width:min(560px,92vw);" +
      "background:#1f2937;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.25);padding:10px 14px;display:none;" +
      "font-family:Inter,sans-serif;color:#fff;font-size:13px";
    label = document.createElement("div");
    label.style.cssText = "margin-bottom:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis";
    var track = document.createElement("div");
    track.style.cssText = "height:8px;background:rgba(255,255,255,.18);border-radius:6px;overflow:hidden";
    fill = document.createElement("div");
    fill.style.cssText = "height:100%;width:0;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);transition:width .6s ease";
    track.appendChild(fill); box.appendChild(label); box.appendChild(track); document.body.appendChild(box);
  }
  async function tick() {
    try {
      var r = await fetch("aiq_progress?t=" + Date.now(), {cache: "no-store"});
      var p = await r.json();
      if (!box && document.body) build();
      if (lastFinal === null) { lastFinal = p.final_n; }
      var deliver = document.getElementById("aiq-deliver");
      if (deliver && p.final_n && p.final_n !== lastFinal && !p.active) { lastFinal = p.final_n; deliver.click(); }
      if (box) {
        if (p.active) {
          box.style.display = "block"; hideAt = 0;
          fill.style.width = Math.round(p.fraction * 100) + "%";
          label.textContent = "⏳ " + p.text + " · " + Math.round(p.elapsed) + "s";
        } else if (box.style.display === "block") {
          if (!hideAt) {
            fill.style.width = "100%";
            label.textContent = (p.video ? "✅ Done in " : "⚠️ Finished after ") + Math.round(p.elapsed) + "s ";
            if (p.video) {
              var link = document.createElement("a");
              link.href = "aiq_image?t=" + Date.now(); link.target = "_blank";
              link.style.cssText = "color:#c7d2fe;font-weight:600";
              link.textContent = "· 🖼️ open / download the full-size image";
              label.appendChild(link);
            } else if (p.message) { label.textContent += "· " + p.message; }
            hideAt = Date.now() + 60000;
          } else if (Date.now() > hideAt) { box.style.display = "none"; }
        }
      }
    } catch (e) {}
    setTimeout(tick, 1000);
  }
  tick();
})();
</script>
"""

HOW_TO = ("**How it works:** describe the whole image in detail and press **🎨 Generate**. To edit, open **🖼️ Pictures to edit**, "
          "add up to 4 pictures and describe the finished result, calling them `<Picture 1>`, `<Picture 2>` and so on. "
          "⚡ Turbo with a 2K (4 MP) size is the fastest; 💎 Quality at 16 MP gives the most detail.")
PROMPT_TIP = ("Long, complete descriptions work best: what the image is (a photo, a painting, a 3D render), the subject, "
              "the setting, the composition and viewpoint, the light and the colors. Put any text to render in quotes. "
              "For edits, say what changes and list what stays exactly the same.")

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=PROGRESS_HEAD, title="MiniMax H3-Image | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown(HOW_TO)

    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label="📝 Prompt (describe the finished image)", lines=10,
                                placeholder="The image is a documentary photograph of ...")
            gr.Markdown(f"<span style='color:#6b7280;font-size:13px'>💡 {PROMPT_TIP}</span>")
        with gr.Column(scale=2):
            size = gr.Dropdown(list(SIZES), value=DEFAULT_SIZE, label="📐 Size",
                               info="4 MP: fastest. 16 MP: the model's best detail")
            speed = gr.Radio(list(SPEEDS), value=DEFAULT_SPEED, label="🚀 Speed",
                             info="Turbo and Balanced use the pDMD few-step LoRA. Quality is the model's own 6-step schedule")
            size_note = gr.Markdown(size_info(DEFAULT_SIZE, EDIT_OWN, None, None, None, None, ATTENTION_DEFAULT, DEFAULT_SPEED))
            seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")
            attention = gr.Radio(ATTENTION_CHOICES, value=ATTENTION_DEFAULT, label="⚡ Attention",
                                 info=("Fast: Comfy Kitchen INT8 attention, quicker on the T4. Exact: full precision"
                                       if ATTENTION["int8"] is not None else "Exact: full precision (Fast is not available on this GPU)"))

    with gr.Accordion("🖼️ Pictures to edit (optional): <Picture 1> to <Picture 4>", open=False):
        edit_mode = gr.Radio([EDIT_OWN, EDIT_SIZE], value=EDIT_OWN, label="Output shape when editing",
                             info="Keep Picture 1's proportions at the selected size's area, or use the selected size as is")
        with gr.Row():
            imgs = [gr.Image(type="pil", label=f"Picture {i + 1}", height=180) for i in range(MAX_IMAGES)]

    with gr.Row():
        gen_btn = gr.Button("🎨 Generate", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    image_out = gr.Image(label="🖼️ Result (full size PNG: use the download button)", type="filepath", format="png", height=560,
                         interactive=False)
    latest_btn = gr.Button("📂 Load latest image (use if the UI lost its connection)")
    status_out = gr.Textbox(label="ℹ️ Status", lines=6, interactive=False)
    deliver_btn = gr.Button("deliver", elem_id="aiq-deliver")  # hidden; clicked by the page script when a result is ready

    _rows, _labels = official_example_rows()
    if _rows:
        gr.Markdown("### 📚 Examples\nMiniMax's own prompts, sizes and seeds. Click one to fill everything above, then press "
                    "**🎨 Generate**.")
        gr.Examples(_rows, inputs=[prompt, size, edit_mode, *imgs, seed, speed], example_labels=_labels, label=None)
    gr.HTML(FOOTER_HTML)

    for _comp in (size, edit_mode, *imgs, attention, speed):
        _comp.change(size_info, [size, edit_mode, *imgs, attention, speed], size_note, queue=False)
    gen_btn.click(start_generation, inputs=[prompt, size, edit_mode, *imgs, seed, attention, speed], outputs=status_out, queue=False)
    deliver_btn.click(deliver_result, outputs=[image_out, status_out], queue=False, show_progress="hidden")
    stop_btn.click(stop_generation, outputs=status_out, queue=False)
    latest_btn.click(load_latest_image, outputs=[image_out, status_out], queue=False)
    clear_btn.click(lambda: (None, "Ready.", "", *([None] * MAX_IMAGES), -1),
                    outputs=[image_out, status_out, prompt, *imgs, seed], queue=False)



# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# Started without preexec_fn: that would force a real fork() of this process (Python otherwise uses vfork), and a
# fork has to copy every pinned page at once. Step 2 stops leftover tunnels before each start. The link works once
# cloudflared has registered a connection with Cloudflare's edge and the random *.trycloudflare.com name is in DNS;
# --protocol http2 because Kaggle often blocks QUIC.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if found.get("late"):  # the ready block was already printed without it
                    print(f"🌐 Cloudflare: {found['url']}")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found


cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, max_threads=8, allowed_paths=[OUTPUT_DIR, EXAMPLES_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _image_route():
    """The newest image as a plain file (opens in the browser at full size, with a download option): no Gradio involved."""
    path = latest_image_path()
    if not path:
        return JSONResponse({"error": "no image yet"}, status_code=404)
    name = os.path.basename(path)
    return FileResponse(path, media_type="image/png", headers={"Content-Disposition": f'inline; filename="{name}"',
                                                               "Cache-Control": "no-store"})


for _path_name, _route in (("/aiq_progress", _progress_route), ("/aiq_image", _image_route)):
    demo.app.router.routes.insert(0, APIRoute(_path_name, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s to come up before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print(f"✅ Ready in {time.time() - _t_load:.0f}s | GPU free {_gb(_free0)} / {_gb(_free1)} | RAM {_ram_in_use()}")
if cf["ready"]:
    print(f"🌐 Cloudflare: {cf['url']}")
else:
    cf["late"] = True
    print("🌐 Cloudflare: warming up, the link appears here in a minute (Gradio works now)")
print(f"🔗 Gradio: {getattr(demo, 'share_url', None) or 'unavailable, use the Cloudflare link'}")
print(f"💾 Images: {OUTPUT_DIR} (latest also at <link>/aiq_image)")
sys.stdout.flush()
while True:  # keep the server process alive; generation logs stream to the notebook cell
    time.sleep(3600)
'''


with open("run_h3_image.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)

import json
if os.path.exists("h3_image_settings.json"):  # Step 1's choices, for a Step 2 re-run after a kernel restart
    with open("h3_image_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Minimal logs: only the engine's own status lines (they start with these markers) and its live progress line reach
# this cell. Library chatter is dropped; Python tracebacks and crash reports are always shown in full.
MARKERS = ("✅", "❌", "⚠️", "🎨", "🌐", "🔗", "💾", "⏹️", "🛑", "🧹", "⚙️", "⏳", "⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏")
CRASH_START = ("Traceback (most recent call last)", "Fatal Python error", "Thread 0x", "Current thread 0x")
_in_crash = [False]


def _keep(part):
    text = part.strip()
    if not text:  # carriage returns / spaces that clear the live line
        return True
    if text.startswith(CRASH_START):
        _in_crash[0] = True
        return True
    if _in_crash[0]:
        if part[:1] in (" ", "\t") or text.startswith(CRASH_START):
            return True
        _in_crash[0] = False  # the exception line closes the traceback
        return True
    return text.startswith(MARKERS)


env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           TF_CPP_MIN_LOG_LEVEL="3", USE_TF="0", USE_FLAX="0", GRADIO_ANALYTICS_ENABLED="False")

print("🚀 Starting MiniMax H3-Image...")
import psutil


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~20 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_h3_image.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass
_kernel_gb = psutil.Process().memory_info().rss / 1024**3
if _kernel_gb > 2.0:
    print(f"⚠️ This notebook kernel holds {_kernel_gb:.1f} GB of RAM. If the engine runs out of memory: Run → Restart, then run Step 2 only.")


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_h3_image.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                        preexec_fn=_die_with_kernel)
import threading
import time
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if _keep(part))
        if pending and pending.strip().startswith(MARKERS):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---